# Module 2 — Analytics Pipeline
## Part A: EDA — `01_eda.ipynb`

**Dataset**: Titanic — loaded once via `sns.load_dataset('titanic')`, immediately saved as `titanic.csv` offline fallback.

**Pipeline order**: Load → Profile → Clean → Univariate → Bivariate → Multivariate → EDA Standardization → Save CSV

> **How to run**: `Kernel → Restart Kernel and Run All Cells`

---

In [7]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install scikit-learn pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [16]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print(f'pandas   version : {pd.__version__}')
print(f'seaborn  version : {sns.__version__}')
print(f'sklearn  version : {sklearn.__version__}')
print(f'numpy    version : {np.__version__}')

pandas   version : 2.2.2
seaborn  version : 0.13.2
sklearn  version : 1.9.1
numpy    version : 2.4.6


## Step 1 — Load Dataset, Profile, Report Missing Values, Save Offline Fallback

In [18]:
# ── Load ONCE from seaborn (network/cache on first run, local cache after) ────
# This is the ONE AND ONLY sns.load_dataset call in this entire module.
# 02_modeling.ipynb reads titanic.csv — it never calls sns.load_dataset again.

df = sns.load_dataset('titanic')

# ── IMMEDIATELY save as offline fallback — this file is committed to the repo ─
df.to_csv('titanic.csv', index=False)

print('Dataset loaded and saved to titanic.csv')
print(f'Shape: {df.shape}')   # expected (891, 15)

Dataset loaded and saved to titanic.csv
Shape: (891, 15)


In [19]:
# ── df.info() ─────────────────────────────────────────────────────────────────
print('── df.info() ──')
df.info()

── df.info() ──
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [20]:
# ── df.describe() ─────────────────────────────────────────────────────────────
print('── df.describe() ──')
display(df.describe())

── df.describe() ──


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [21]:
# ── Missing value profile ──────────────────────────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct':   missing_pct
})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print('── Columns with missing values ──')
print(missing_report.to_string())
print()

# ── Threshold rule applied to each column ─────────────────────────────────────
print('── Threshold rule decisions ──')
print('Column        | Missing %  | Rule              | Decision')
print('-' * 70)
for col, row in missing_report.iterrows():
    pct = row['missing_pct']
    if pct < 5:
        rule     = '< 5%  → drop rows'
        decision = 'Drop those rows'
    elif pct <= 30:
        rule     = '5-30% → impute'
        decision = 'Median imputation'
    else:
        rule     = '> 30% → drop column'
        decision = 'Drop column'
    print(f'{col:<14}| {pct:>9.2f}% | {rule:<18} | {decision}')

── Columns with missing values ──
             missing_count  missing_pct
deck                   688        77.22
age                    177        19.87
embarked                 2         0.22
embark_town              2         0.22

── Threshold rule decisions ──
Column        | Missing %  | Rule              | Decision
----------------------------------------------------------------------
deck          |     77.22% | > 30% → drop column | Drop column
age           |     19.87% | 5-30% → impute     | Median imputation
embarked      |      0.22% | < 5%  → drop rows  | Drop those rows
embark_town   |      0.22% | < 5%  → drop rows  | Drop those rows


In [22]:
# ── Assertions: confirm shape and expected columns ─────────────────────────────
assert df.shape == (891, 15), f'Unexpected shape: {df.shape}'
assert 'survived' in df.columns
assert 'age'      in df.columns
assert 'fare'     in df.columns

import os
assert os.path.exists('titanic.csv'), 'titanic.csv was not saved!'

print(f'✅ STEP 1 COMPLETE')
print(f'   Shape           : {df.shape}')
print(f'   titanic.csv     : saved ({os.path.getsize("titanic.csv")} bytes)')
print(f'   Columns w/ NaN  : {len(missing_report)}')

✅ STEP 1 COMPLETE
   Shape           : (891, 15)
   titanic.csv     : saved (57910 bytes)
   Columns w/ NaN  : 4
